# TP – Systèmes de recommandation

**Cours** : INF5063 – Machine learning : applications (G. Nollet, L. Benedetti)
**Auteur** : Alban Rouault

Objectif : concevoir et évaluer un système de recommandation de films par filtrage collaboratif
*user-based* sur le jeu de données Kaggle
[The Movies Dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset).

## Imports et configuration

Toutes les dépendances sont déclarées dans `pyproject.toml` et gérées avec `uv`
(`uv run jupyter lab` pour lancer le notebook).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Affichage des DataFrames
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 60)

# Reproductibilité (séparation train/test, tirages aléatoires)
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# Emplacement des données
DATA_DIR = Path("Ressources/dataset")

## Exercice 1 – Chargement des données

### 1) Chargement des fichiers

Le dataset a été récupéré sur Kaggle et décompressé dans `Ressources/dataset/`.
Seuls trois fichiers sont utilisés dans ce TP :

| Fichier | Contenu |
|---|---|
| `movies_metadata.csv` | Métadonnées des films (titre, genres, date de sortie, budget, recettes…), identifiés par leur id TMDB |
| `ratings_small.csv` | 100 000 notes d'utilisateurs, films identifiés par leur id MovieLens |
| `links_small.csv` | Table de correspondance entre les id MovieLens, IMDB et TMDB |

Remarque : `movies_metadata.csv` contient des colonnes au typage hétérogène (la colonne `id`
mélange entiers et chaînes sur quelques lignes mal formées). On passe `low_memory=False` pour que
pandas lise le fichier d'un bloc et infère un type cohérent par colonne ; le nettoyage des id sera
fait à la question 5.

In [ ]:
movies = pd.read_csv(DATA_DIR / "movies_metadata.csv", low_memory=False)
ratings = pd.read_csv(DATA_DIR / "ratings_small.csv")
links = pd.read_csv(DATA_DIR / "links_small.csv")

tables = {"movies_metadata": movies, "ratings_small": ratings, "links_small": links}
for name, df in tables.items():
    print(f"{name:<16} {df.shape[0]:>7,} lignes  x {df.shape[1]:>2} colonnes")